In [0]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.ml import Pipeline
from pyspark.ml.classification import NaiveBayes, RandomForestClassifier 
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

# 1. SETUP & DATA LOADING
output_dir = "../../output/graphs"
os.makedirs(output_dir, exist_ok=True)

df = spark.table("workspace.capstone_project.nyc_model_ready")
df = df.filter(df.delay_indicator.isNotNull())

numeric_features = ['hour', 'day_of_week', 'month', 'year', 'unified_alarm_level', 'calls_past_30min', 'calls_past_60min']
categorical_features = ['season', 'incident_category', 'unified_call_source', 'location_area']

# 2. PREPROCESSING PIPELINE
stages = []
for col in categorical_features:
    indexer = StringIndexer(inputCol=col, outputCol=f"{col}_index", handleInvalid="keep")
    encoder = OneHotEncoder(inputCols=[f"{col}_index"], outputCols=[f"{col}_vec"])
    stages += [indexer, encoder]

assembler_inputs = [f"{col}_vec" for col in categorical_features] + numeric_features
assembler = VectorAssembler(inputCols=assembler_inputs, outputCol="unscaled_features")
stages.append(assembler)

scaler = StandardScaler(inputCol="unscaled_features", outputCol="features")
stages.append(scaler)

# 3. MODEL TRAINING
train_df, test_df = df.randomSplit([0.8, 0.2], seed=42)

# Baseline Model: Naive Bayes
nb = NaiveBayes(labelCol="delay_indicator", featuresCol="features", smoothing=1.0)
pipeline_nb = Pipeline(stages=stages + [nb])
model_nb = pipeline_nb.fit(train_df)
preds_nb = model_nb.transform(test_df)

# Proposed Model: Random Forest
rf = RandomForestClassifier(labelCol="delay_indicator", featuresCol="features", numTrees=100, seed=42)
pipeline_rf = Pipeline(stages=stages + [rf])
model_rf = pipeline_rf.fit(train_df)
preds_rf = model_rf.transform(test_df)

# 4. EVALUATION
def get_metrics(predictions, model_name):
    auc_eval = BinaryClassificationEvaluator(labelCol="delay_indicator", metricName="areaUnderROC")
    multi_eval = MulticlassClassificationEvaluator(labelCol="delay_indicator", predictionCol="prediction")
    
    return {
        "Model": model_name,
        "AUC-ROC": round(auc_eval.evaluate(predictions), 3),
        "Precision": round(multi_eval.evaluate(predictions, {multi_eval.metricName: "weightedPrecision"}), 3),
        "Recall": round(multi_eval.evaluate(predictions, {multi_eval.metricName: "weightedRecall"}), 3)
    }

# Updated results list
results = [get_metrics(preds_nb, "Naive Bayes"), get_metrics(preds_rf, "Random Forest")]
comparison_df = pd.DataFrame(results)

print("\nBaseline (NB) vs Proposed (RF) Comparison")
print(comparison_df.to_string(index=False))